# 02 — Feature Engineering
## Dynamic Pricing & Demand Forecasting
### Goal: Build all features needed for LightGBM from the master dataset

## Section 1: Build Master Dataset
Goal: Merge all 3 raw files into single df with 58M rows

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load raw files
sales    = pd.read_csv('../data/raw/sales_train_validation.csv')
calendar = pd.read_csv('../data/raw/calendar.csv')
prices   = pd.read_csv('../data/raw/sell_prices.csv')

print("Sales shape:", sales.shape)
print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)
print("\nMemory usage:")
print(f"  Sales:    {sales.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Calendar: {calendar.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Prices:   {prices.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("\nAll files loaded successfully!")

### 1.1 Melt + Merge

In [ ]:
# Melt sales wide to long
day_cols = [c for c in sales.columns if c.startswith('d_')]

df = sales.melt(
    id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    value_vars=day_cols,
    var_name='day_id',
    value_name='sales'
)

del sales
print("After melt:", df.shape)

# Merge with calendar
df = df.merge(
    calendar[['d', 'date', 'wm_yr_wk', 'weekday', 'wday',
              'month', 'year', 'event_name_1', 'event_type_1',
              'snap_CA', 'snap_TX', 'snap_WI']],
    left_on='day_id',
    right_on='d',
    how='left'
)

df.drop(columns=['d'], inplace=True)
df['date'] = pd.to_datetime(df['date'])
print("After calendar merge:", df.shape)

# Merge with prices
df = df.merge(
    prices,
    on=['item_id', 'store_id', 'wm_yr_wk'],
    how='left'
)

print("After price merge:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSample:")
print(df[['item_id', 'store_id', 'date', 'sales', 'sell_price']].head(3))

### 1.2 Null check

In [ ]:
print("Null counts:")
print(df.isnull().sum())
print(f"\nTotal nulls: {df.isnull().sum().sum():,}")

### 1.3 Handle null values

#### event_name_1 and event_type_1 nulls
NULL means normal day — not missing data

In [ ]:
# Fill event nulls with 'none' — NULL means normal day
df['event_name_1'] = df['event_name_1'].fillna('none')
df['event_type_1'] = df['event_type_1'].fillna('none')

print("Event nulls after fix:")
print(df[['event_name_1', 'event_type_1']].isnull().sum())

#### sell_price nulls
Diagnosis: 100% of null price rows have zero sales
Meaning: item did not exist in store that week (pre-launch or discontinued)
Decision: fill with 0 — honest signal that item was unavailable

In [ ]:
# Diagnose first
null_mask = df['sell_price'].isnull()
null_df = df[null_mask]

print("=== NULL PRICE DIAGNOSIS ===")
print(f"Total null price rows:          {null_mask.sum():,}")
print(f"Zero sales + null price:        {(null_df['sales']==0).sum():,}")
print(f"Non-zero sales + null price:    {(null_df['sales']>0).sum():,}")
print(f"% null price rows with 0 sales: {(null_df['sales']==0).mean()*100:.1f}%")

In [ ]:
# Fix — fill with 0 (item unavailable signal)
df['sell_price'] = df['sell_price'].fillna(0)

# Verify
print("Nulls after fix:")
print(df['sell_price'].isnull().sum())
print(f"\nZero price rows: {(df['sell_price']==0).sum():,}")
print(f"Non-zero price rows: {(df['sell_price']>0).sum():,}")

#### Final null check

In [ ]:
print("Final null count across all columns:")
print(df.isnull().sum())
print(f"\nTotal nulls remaining: {df.isnull().sum().sum()}")

## Section 2: Calendar Features
Goal: Extract temporal signals from date column for LightGBM
These features capture seasonality, day-of-week patterns, and event effects

### 2.1 Basic date features

In [ ]:
# Basic date features
df['day_of_week']    = df['date'].dt.dayofweek        # 0=Monday, 6=Sunday
df['day_of_month']   = df['date'].dt.day              # 1-31
df['week_of_year']   = df['date'].dt.isocalendar().week.astype(int)  # 1-52
df['quarter']        = df['date'].dt.quarter          # 1-4
df['is_weekend']     = (df['day_of_week'] >= 5).astype(int)  # 1=weekend
df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
df['is_month_end']   = df['date'].dt.is_month_end.astype(int)

print("Date features added:")
print(df[['date', 'day_of_week', 'day_of_month',
          'week_of_year', 'quarter',
          'is_weekend', 'is_month_start',
          'is_month_end']].head(5))

### 2.2 SNAP feature
Collapse 3 state SNAP columns into 1 relevant column per store
EDA finding: SNAP drives +12.7% demand lift concentrated in FOODS

In [ ]:
# Create single is_snap feature based on store state
df['is_snap'] = np.where(
    df['state_id'] == 'CA', df['snap_CA'],
    np.where(
        df['state_id'] == 'TX', df['snap_TX'],
        df['snap_WI']
    )
)

# Drop original 3 SNAP columns — no longer needed
df.drop(columns=['snap_CA', 'snap_TX', 'snap_WI'], inplace=True)

print("is_snap created:")
print(df['is_snap'].value_counts())
print(f"\nSNAP days: {df['is_snap'].mean()*100:.1f}% of all rows")

In [ ]:
df.head()

### 2.3 Event features
- Sporting events only type above baseline (+3.8%) — SuperBowl snack buying effect
- National events biggest drop (-14.6%) — store closures on major holidays
- Pre-holiday demand spike hypothesised — days_before_holiday feature 
  to be built later in this notebook

In [ ]:
# Event binary features
df['is_event']     = (df['event_name_1'] != 'none').astype(int)
df['is_sporting']  = (df['event_type_1'] == 'Sporting').astype(int)
df['is_national']  = (df['event_type_1'] == 'National').astype(int)
df['is_religious'] = (df['event_type_1'] == 'Religious').astype(int)
df['is_cultural']  = (df['event_type_1'] == 'Cultural').astype(int)

print("Event features added:")
print(df[['is_event', 'is_sporting',
          'is_national', 'is_religious',
          'is_cultural']].sum())

### 2.4 Drop redundant calendar columns
Keep only engineered features — remove raw columns no longer needed

In [ ]:
# Drop raw columns we've extracted features from
df.drop(columns=[
    'day_id',        # replaced by date features
    'weekday',       # replaced by day_of_week
    'wday',          # duplicate of day_of_week
    'event_name_1',  # replaced by is_event flags
    'event_type_1',  # replaced by event type flags
], inplace=True)

print("Columns after calendar features:")
print(df.columns.tolist())
print(f"\nShape: {df.shape}")

### 2.5 Verify calendar features

In [ ]:
print("Sample rows with all calendar features:")
print(df[['date', 'day_of_week', 'is_weekend',
          'month', 'quarter', 'is_snap',
          'is_event', 'is_sporting']].head(7))

print(f"\nWeekend rows: {df['is_weekend'].sum():,} ({df['is_weekend'].mean()*100:.1f}%)")
print(f"Event rows:   {df['is_event'].sum():,} ({df['is_event'].mean()*100:.1f}%)")
print(f"SNAP rows:    {df['is_snap'].sum():,} ({df['is_snap'].mean()*100:.1f}%)")

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

### 3.1 Sort data correctly
Must sort by item, store, date before any price calculations

In [ ]:
# Sort by item, store, date — critical for all lag/price calculations
df = df.sort_values(
    ['item_id', 'store_id', 'date']
).reset_index(drop=True)

print("Sorted by item_id, store_id, date")
print(df[['item_id', 'store_id', 'date', 'sell_price']].head(10))

### 3.2 Price change features
Capture how price changed relative to recent history

In [ ]:
# Group by item + store for all price calculations
grp = df.groupby(['item_id', 'store_id'])['sell_price']

# Price 7 days ago
df['price_lag_7']  = grp.transform(lambda x: x.shift(7))

# Price 28 days ago
df['price_lag_28'] = grp.transform(lambda x: x.shift(28))

# % change vs 7 days ago
df['price_change_7'] = (
    (df['sell_price'] - df['price_lag_7']) / 
    (df['price_lag_7'] + 1e-8)
)

# % change vs 28 days ago
df['price_change_28'] = (
    (df['sell_price'] - df['price_lag_28']) / 
    (df['price_lag_28'] + 1e-8)
)

print("Price change features added:")
print(df[['item_id', 'store_id', 'date', 
          'sell_price', 'price_lag_7',
          'price_change_7', 'price_change_28']].head(14))

### 3.3 Price rolling average
Capture price trend over time

In [ ]:
# Rolling mean price — 28 day window
df['price_rolling_mean_28'] = grp.transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean()
)

# Price relative to its own 28-day average
# > 1 means price went UP, < 1 means price went DOWN
df['price_vs_rolling_mean'] = (
    df['sell_price'] / 
    (df['price_rolling_mean_28'] + 1e-8)
)

print("Price rolling features added:")
print(df[['item_id', 'store_id', 'date',
          'sell_price', 'price_rolling_mean_28',
          'price_vs_rolling_mean']].head(35))

### 3.4 Price rank within category
Is this item cheap or expensive relative to its category?

In [ ]:
# Average price per item (across all time)
item_avg_price = df.groupby(
    'item_id'
)['sell_price'].mean().rename('item_avg_price')

df = df.join(item_avg_price, on='item_id')

# Average price per category
cat_avg_price = df.groupby(
    'cat_id'
)['sell_price'].mean().rename('cat_avg_price')

df = df.join(cat_avg_price, on='cat_id')

# Price rank — is this item cheap or expensive in its category?
df['price_vs_cat_avg'] = (
    df['item_avg_price'] / 
    (df['cat_avg_price'] + 1e-8)
)

print("Price rank features added:")
print(df.groupby('cat_id')[['item_avg_price', 
                             'cat_avg_price',
                             'price_vs_cat_avg']].mean().round(3))

In [ ]:
# show variation at item level — not category level
print(df.groupby('item_id')['price_vs_cat_avg'].mean().describe())

### 3.5 Verify price features

In [ ]:
price_features = [
    'sell_price', 'price_lag_7', 'price_lag_28',
    'price_change_7', 'price_change_28',
    'price_rolling_mean_28', 'price_vs_rolling_mean',
    'item_avg_price', 'cat_avg_price', 'price_vs_cat_avg'
]

print("Price features null counts:")
print(df[price_features].isnull().sum())
print(f"\nShape: {df.shape}")

## Section 4: Lag Features
Goal: Capture each item's own recent sales history
These are the most important features in the model
Each lag is calculated independently per item-store combination

### 4.1 Sales lag features
* lag_7  → same day last week
* lag_28 → same day last month
* lag_56 → same day 2 months ago

In [ ]:
# Group by item + store — all lags calculated independently
grp = df.groupby(['item_id', 'store_id'])['sales']

# Lag features
df['lag_7']  = grp.transform(lambda x: x.shift(7))
df['lag_28'] = grp.transform(lambda x: x.shift(28))
df['lag_56'] = grp.transform(lambda x: x.shift(56))

print("Lag features added:")
print(df[['item_id', 'store_id', 'date', 
          'sales', 'lag_7', 'lag_28', 
          'lag_56']].head(35))

### 4.2 Verify lag features

In [ ]:
print("Lag feature null counts:")
print(df[['lag_7', 'lag_28', 'lag_56']].isnull().sum())

print(f"\nExpected nulls:")
print(f"lag_7  → {30490 * 7:,}  (30,490 items × 7 days)")
print(f"lag_28 → {30490 * 28:,} (30,490 items × 28 days)")
print(f"lag_56 → {30490 * 56:,} (30,490 items × 56 days)")

print(f"\nShape: {df.shape}")

## Section 5: Rolling Features
Goal: Capture recent demand trends and volatility per item-store
Rolling features smooth out daily noise and capture baseline demand

In [ ]:
# Group by item + store
grp = df.groupby(['item_id', 'store_id'])['sales']

# Rolling mean features
# shift(1) ensures we don't include today's sales (data leakage)
df['rmean_7']  = grp.transform(
    lambda x: x.shift(1).rolling(7,  min_periods=1).mean()
)
df['rmean_28'] = grp.transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean()
)
df['rmean_56'] = grp.transform(
    lambda x: x.shift(1).rolling(56, min_periods=1).mean()
)

print("Rolling mean features added:")
print(df[['item_id', 'store_id', 'date',
          'sales', 'rmean_7',
          'rmean_28', 'rmean_56']].head(35))

### 5.2 Rolling std features
Demand volatility — how consistent is this item's sales?
High std → unpredictable item → harder to forecast
Low std  → stable item → easier to forecast

In [ ]:
# Rolling std — demand volatility
df['rstd_7']  = grp.transform(
    lambda x: x.shift(1).rolling(7,  min_periods=2).std()
)
df['rstd_28'] = grp.transform(
    lambda x: x.shift(1).rolling(28, min_periods=2).std()
)

print("Rolling std features added:")
print(df[['item_id', 'store_id', 'date',
          'sales', 'rstd_7', 'rstd_28']].head(35))

### 5.3 Rolling max and min features
Capture demand spikes and floors

In [ ]:
# Rolling max — peak demand signal
df['rmax_7']  = grp.transform(
    lambda x: x.shift(1).rolling(7,  min_periods=1).max()
)
df['rmax_28'] = grp.transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).max()
)

# Rolling min — floor demand signal
df['rmin_7']  = grp.transform(
    lambda x: x.shift(1).rolling(7,  min_periods=1).min()
)

df['rmin_28'] = grp.transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).min()
)

print("Rolling max/min features added:")
print(df[['item_id', 'store_id', 'date',
          'sales', 'rmax_7', 'rmax_28',
          'rmin_7']].head(35))

### 5.4 Verify all rolling features

In [ ]:
rolling_features = [
    'rmean_7', 'rmean_28', 'rmean_56',
    'rstd_7', 'rstd_28',
    'rmax_7', 'rmax_28', 'rmin_7', 'rmin_28'
]

print("Rolling feature null counts:")
print(df[rolling_features].isnull().sum())
print(f"\nShape: {df.shape}")
print(f"\nTotal features built: {df.shape[1]}")

- rmean_7  → average sales last 7 days — short term demand baseline
- rmean_28 → average sales last 28 days — monthly demand baseline
- rmean_56 → average sales last 56 days — long term stable baseline
- rstd_7   → how much sales varied last 7 days — recent volatility
- rstd_28  → how much sales varied last 28 days — longer term volatility
- rmax_7   → highest single day sales last 7 days — recent demand ceiling
- rmax_28  → highest single day sales last 28 days — monthly demand ceiling
- rmin_7   → lowest single day sales last 7 days — recent demand floor
- rmin_28  → lowest single day sales last 28 days — monthly demand floor

## Section 6: Additional Features

### 6.1 Interaction Features

In [ ]:
# ── 6.1 Interaction Features ─────────────────────────────

# SNAP × FOODS — EDA showed +17.2% lift in FOODS vs +2-3% in other categories
df['is_snap_foods'] = (
    (df['is_snap'] == 1) & (df['cat_id'] == 'FOODS')
).astype('int8')

print(f"is_snap_foods distribution:")
print(df['is_snap_foods'].value_counts())
print(f"\nSNAP FOODS rows: {df['is_snap_foods'].sum():,}")

### 6.2 Holiday Features

In [ ]:
# ── 6.2 Holiday Features ─────────────────────────────────

# get all event dates
event_dates = df[df['is_event'] == 1]['date'].unique()
event_dates = pd.DatetimeIndex(sorted(event_dates))

# days until next holiday for each date
def days_to_next_event(date):
    future = event_dates[event_dates > date]
    if len(future) == 0:
        return 999
    return (future[0] - date).days

# apply to unique dates first then map — faster than row by row
unique_dates = df['date'].unique()
date_to_days = {d: days_to_next_event(pd.Timestamp(d)) for d in unique_dates}
df['days_before_holiday'] = df['date'].map(date_to_days).astype('int16')

# pre-holiday flag — within 3 days of a holiday
df['is_pre_holiday'] = (
    (df['days_before_holiday'] > 0) & 
    (df['days_before_holiday'] <= 3)
).astype('int8')

print(f"days_before_holiday sample:")
print(df[['date', 'days_before_holiday', 
          'is_pre_holiday']].drop_duplicates('date').head(10))
print(f"\nPre-holiday rows: {df['is_pre_holiday'].sum():,}")
print(f"Expected: 3 days × 162 events × 30,490 series = {3*162*30490:,}")

### 6.3 Aggregated Demand Features

In [ ]:
# ── 6.3 Aggregated Demand Features ───────────────────────
# Rolling mean at higher aggregation levels
# Captures cross-series demand signal — used by top M5 competitors

# store level — avg demand across all items in this store last 28 days
store_grp = df.groupby(['store_id', 'date'])['sales'].mean().reset_index()
store_grp.columns = ['store_id', 'date', 'store_daily_avg']
store_grp['rmean_28_store'] = store_grp.groupby('store_id')['store_daily_avg'].transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean()
)
df = df.merge(store_grp[['store_id', 'date', 'rmean_28_store']], 
              on=['store_id', 'date'], how='left')

# category level — avg demand across all items in this category last 28 days
cat_grp = df.groupby(['cat_id', 'date'])['sales'].mean().reset_index()
cat_grp.columns = ['cat_id', 'date', 'cat_daily_avg']
cat_grp['rmean_28_cat'] = cat_grp.groupby('cat_id')['cat_daily_avg'].transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean()
)
df = df.merge(cat_grp[['cat_id', 'date', 'rmean_28_cat']], 
              on=['cat_id', 'date'], how='left')

# department level — avg demand across all items in this dept last 28 days
dept_grp = df.groupby(['dept_id', 'date'])['sales'].mean().reset_index()
dept_grp.columns = ['dept_id', 'date', 'dept_daily_avg']
dept_grp['rmean_28_dept'] = dept_grp.groupby('dept_id')['dept_daily_avg'].transform(
    lambda x: x.shift(1).rolling(28, min_periods=1).mean()
)
df = df.merge(dept_grp[['dept_id', 'date', 'rmean_28_dept']], 
              on=['dept_id', 'date'], how='left')

print(f"Aggregated features added:")
print(df[['rmean_28_store', 'rmean_28_cat', 
          'rmean_28_dept']].isnull().sum())
print(f"\nShape: {df.shape}")

### Section 5.5 — Memory optimization & categorical encoding
Reduce memory from 22.8 GB before saving to disk
Downcast dtypes without losing information

In [ ]:
# Section 7: Memory Optimisation
print(f"Memory before: {df.memory_usage(deep=True).sum() / 1e9:.1f} GB")

cat_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
for col in cat_cols:
    df[col] = df[col].astype('category')

binary_cols = [
    'is_weekend', 'is_month_start', 'is_month_end',
    'is_snap', 'is_event', 'is_sporting', 'is_national',
    'is_religious', 'is_cultural',
    'is_snap_foods', 'is_pre_holiday'
]
for col in binary_cols:
    df[col] = df[col].astype('int8')

small_int_cols = {
    'day_of_week':         'int8',
    'day_of_month':        'int8',
    'week_of_year':        'int8',
    'quarter':             'int8',
    'month':               'int8',
    'year':                'int16',
    'wm_yr_wk':            'int32',
    'days_before_holiday': 'int16',
}
for col, dtype in small_int_cols.items():
    df[col] = df[col].astype(dtype)

print(f"Max sales value: {df['sales'].max()}")
df['sales'] = df['sales'].astype('int16')

float_cols = df.select_dtypes('float64').columns.tolist()
for col in float_cols:
    df[col] = df[col].astype('float32')

print(f"Memory after:  {df.memory_usage(deep=True).sum() / 1e9:.1f} GB")
print(df.dtypes.value_counts())
print(f"\nShape: {df.shape}")

## Section 6: Save Processed Dataset
Goal: Save df_train to data/processed/ for use in modeling notebooks

In [ ]:
import os
import pickle

# Create processed folder if not exists
os.makedirs('../data/processed', exist_ok=True)

# Save as pickle
df.to_pickle('../data/processed/df_train.pkl')

# Verify
saved = pd.read_pickle('../data/processed/df_train.pkl')
print(f"Saved shape:  {df.shape}")
print(f"Loaded shape: {saved.shape}")
print(f"\nFile size: {os.path.getsize('../data/processed/df_train.pkl') / 1e9:.2f} GB")